# Track C — Data Quality Check & Scaler Preparation
**Project:** VOLTA — Energy Pressure Forecasting  
**Goal:** Conduct rigorous data quality checks, handle missing values, and prepare leak-free feature scaling on the training split.

In [1]:
# استيراد مكتبات التعامل مع البيانات والتحجيم
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

print('Track C Libraries Imported Successfully!')

Track C Libraries Imported Successfully!


## 1. Load Data & Verify Timeline Completeness
Load the dataset and sort chronologically to maintain time-series integrity.

In [2]:
# 1. قراءة البيانات المدمجة والمنظفة
df = pd.read_csv('../data/processed/volta_clean_hourly.csv')

# 2. ضبط التوقيت الزمني
df['time'] = pd.to_datetime(df['time'], utc=True)
df = df.sort_values('time').set_index('time')

# 3. عرض ملخص النقص لـ الأهداف الأربعة المعتمدة
print(f'Total Hourly Rows: {len(df)}')
print('Missing values in target columns:')
print(
    df[[
        'price actual',
        'total load actual',
        'renewable_total',
        'fossil_total',
    ]]
    .isnull()
    .sum()
)

Total Hourly Rows: 35064
Missing values in target columns:
price actual         0
total load actual    0
renewable_total      0
fossil_total         0
dtype: int64


## 2. Handle Missing Values
Apply forward filling (`ffill`) for small gaps in time-series data to ensure continuity without using future information.

In [3]:
# 1. تحديد أعمدة الأهداف الرئيسية
features_to_scale = [
    'price actual',
    'total load actual',
    'renewable_total',
    'fossil_total',
]

# 2. أخذ نسخة ومعالجة القيم المفقودة للأعمدة المستهدفة فقط
df_cleaned = df[features_to_scale].copy()
df_cleaned = df_cleaned.ffill().bfill()

# 3. التأكد من اختفاء جميع القيم المفقودة تماماً)
print(
    'Remaining Missing Values in Target Features:',
    df_cleaned.isnull().sum().sum(),
)

Remaining Missing Values in Target Features: 0


## 3. Chronological Train / Test Split
Split the data chronologically into:
- **Train Set:** 2015-01-01 to 2016-12-31
- **Validation set:** 2017-01-01 to 2017-12-31
- **Test Set:** 2018-01-01 to 2018-12-31 (Kept strictly untouched)

In [6]:
# 1. التقسيم الزمني المعياري الخالي من التسريب (2015-2016 تدريب | 2017 تحقق | 2018 اختبار)
train_data = df_cleaned.loc['2015-01-01':'2016-12-31']
val_data   = df_cleaned.loc['2017-01-01':'2017-12-31']
test_data  = df_cleaned.loc['2018-01-01':'2018-12-31']

print(f"Train Period:      {train_data.index.min()} to {train_data.index.max()} (Rows: {len(train_data)})")
print(f"Validation Period: {val_data.index.min()} to {val_data.index.max()} (Rows: {len(val_data)})")
print(f"Test Period:       {test_data.index.min()} to {test_data.index.max()} (Rows: {len(test_data)})")

Train Period:      2015-01-01 00:00:00+00:00 to 2016-12-31 23:00:00+00:00 (Rows: 17544)
Validation Period: 2017-01-01 00:00:00+00:00 to 2017-12-31 23:00:00+00:00 (Rows: 8760)
Test Period:       2018-01-01 00:00:00+00:00 to 2018-12-31 22:00:00+00:00 (Rows: 8759)


## 4. Leak-Free Feature Scaling
Fit the `MinMaxScaler` **strictly on the training data**, then transform both train and test sets independently.

In [16]:
# 1. إنشاء أداة التحجيم
scaler = MinMaxScaler(feature_range=(0, 1))

# 2. ملاءمة الـ Scaler على بيانات التدريب (2015-2016) فقط لمنع التسريب
train_scaled = scaler.fit_transform(train_data[features_to_scale])

# 3. تحويل بيانات التحقق والاختبار بنفس أداة التدريب
val_scaled  = scaler.transform(val_data[features_to_scale])
test_scaled = scaler.transform(test_data[features_to_scale])

# 4. تحويل النتائج إلى DataFrames
train_scaled_df = pd.DataFrame(train_scaled, columns=features_to_scale, index=train_data.index)
val_scaled_df   = pd.DataFrame(val_scaled,   columns=features_to_scale, index=val_data.index)
test_scaled_df  = pd.DataFrame(test_scaled,  columns=features_to_scale, index=test_data.index)

print("Scaling completed cleanly across Train, Validation, and Test sets!")
print('=' * 65)
print('Chronological Data Split & Timeline Verification:')
print('=' * 65)
print(
    f'Train Period:      {train_data.index.min()} to {train_data.index.max()}  (Rows: {len(train_data)})'
)
print(
    f'Validation Period: {val_data.index.min()} to {val_data.index.max()}  (Rows: {len(val_data)})'
)
print(
    f'Test Period:       {test_data.index.min()} to {test_data.index.max()}  (Rows: {len(test_data)})'
)
print('=' * 65)
print('Track C Scaling completed cleanly without Data Leakage!')

Scaling completed cleanly across Train, Validation, and Test sets!
Chronological Data Split & Timeline Verification:
Train Period:      2015-01-01 00:00:00+00:00 to 2016-12-31 23:00:00+00:00  (Rows: 17544)
Validation Period: 2017-01-01 00:00:00+00:00 to 2017-12-31 23:00:00+00:00  (Rows: 8760)
Test Period:       2018-01-01 00:00:00+00:00 to 2018-12-31 22:00:00+00:00  (Rows: 8759)
Track C Scaling completed cleanly without Data Leakage!


## 5. Sliding Window Integrity Verification (No Data Leakage Example)

Create a hand-crafted small example window (Lookback = 168 hours, Horizon = 24 hours) to explicitly confirm that all input timestamps strictly precede target timestamps.

In [17]:
LOOKBACK = 168

# دمج آخر 168 ساعة من التدريب قبل بداية الفاليديشن لمنع ضياع أول 7 أيام
val_padded_raw = pd.concat([train_scaled_df.iloc[-LOOKBACK:], val_scaled_df])
val_scaled = scaler.transform(val_padded_raw)

test_padded_raw = pd.concat([val_scaled_df.iloc[-LOOKBACK:], test_scaled_df])
test_scaled = scaler.transform(test_padded_raw)

print("=" * 80)
print("TRACK C: CHRONOLOGICAL DATA SPLIT & LOOKBACK TIMELINE VERIFICATION")
print("=" * 80)

# 1. فترة التدريب (Train)
print("\n1. TRAIN SET (2015 - 2016):")
print(f"   • Start Timestamp : {train_scaled_df.index.min()}")
print(f"   • End Timestamp   : {train_scaled_df.index.max()}")
print(f"   • Total Hours     : {len(train_scaled_df)} hours")

# 2. فترة التحقق (Validation with Padding)
print("\n2. VALIDATION SET (2017 with 168h Lookback Padding):")
print(f"   • Lookback Start  : {val_padded_raw.index.min()}  <-- (Late Dec 2016)")
print(f"   • First Target    : {val_scaled_df.index.min()}  <-- (Exactly Jan 1st, 2017)")
print(f"   • Validation End  : {val_scaled_df.index.max()}")
print(f"   • Total Rows      : {len(val_padded_raw)} hours (168h Lookback + {len(val_scaled_df)}h Val)")

# 3. فترة الاختبار (Test with Padding)
print("\n3. TEST SET (2018 with 168h Lookback Padding):")
print(f"   • Lookback Start  : {test_padded_raw.index.min()}  <-- (Late Dec 2017)")
print(f"   • First Target    : {test_scaled_df.index.min()}  <-- (Exactly Jan 1st, 2018)")
print(f"   • Test End        : {test_scaled_df.index.max()}")
print(f"   • Total Rows      : {len(test_padded_raw)} hours (168h Lookback + {len(test_scaled_df)}h Test)")

print("=" * 80)
print("STATUS: Scaler fitted ONLY on Train set. Lookback padded cleanly without data leakage!")
print("=" * 80)

TRACK C: CHRONOLOGICAL DATA SPLIT & LOOKBACK TIMELINE VERIFICATION

1. TRAIN SET (2015 - 2016):
   • Start Timestamp : 2015-01-01 00:00:00+00:00
   • End Timestamp   : 2016-12-31 23:00:00+00:00
   • Total Hours     : 17544 hours

2. VALIDATION SET (2017 with 168h Lookback Padding):
   • Lookback Start  : 2016-12-25 00:00:00+00:00  <-- (Late Dec 2016)
   • First Target    : 2017-01-01 00:00:00+00:00  <-- (Exactly Jan 1st, 2017)
   • Validation End  : 2017-12-31 23:00:00+00:00
   • Total Rows      : 8928 hours (168h Lookback + 8760h Val)

3. TEST SET (2018 with 168h Lookback Padding):
   • Lookback Start  : 2017-12-25 00:00:00+00:00  <-- (Late Dec 2017)
   • First Target    : 2018-01-01 00:00:00+00:00  <-- (Exactly Jan 1st, 2018)
   • Test End        : 2018-12-31 22:00:00+00:00
   • Total Rows      : 8927 hours (168h Lookback + 8759h Test)
STATUS: Scaler fitted ONLY on Train set. Lookback padded cleanly without data leakage!
